In [3]:

import json, io, requests
import numpy as np
from PIL import Image
from scipy.ndimage import label, binary_dilation

BASE_DIR = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud"

WHITE_THR = 218
IMG_SIZE, PAD = (1200, 1200), 80
WEB_HDR = {"User-Agent": "Mozilla/5.0 Chrome/120.0"}

def make_black_bg_fixed(raw: bytes):
    img = Image.open(io.BytesIO(raw)).convert("RGBA")
    arr = np.array(img, dtype=np.uint8).copy()

    transparent = (arr[:,:,3] == 0)
    white_candidate = (
        (arr[:,:,0] >= WHITE_THR) &
        (arr[:,:,1] >= WHITE_THR) &
        (arr[:,:,2] >= WHITE_THR) &
        (arr[:,:,3] > 0)
    )
    labeled, n = label(white_candidate)
    print(f"Obraz: {img.size}, alpha=0 px: {transparent.sum()}, jasne px: {white_candidate.sum()}, komponenty: {n}")

    if n > 0:
        dilated_transparent = binary_dilation(transparent, iterations=2)
        touching = dilated_transparent & white_candidate
        edge_labels = set(labeled[touching].tolist())
        edge_labels.discard(0)

        h, w = arr.shape[:2]
        border_labels = set()
        border_labels.update(labeled[0, :].tolist())
        border_labels.update(labeled[h-1, :].tolist())
        border_labels.update(labeled[:, 0].tolist())
        border_labels.update(labeled[:, w-1].tolist())
        border_labels.discard(0)

        bg_labels = edge_labels | border_labels
        print(f"Komponenty tła: {len(bg_labels)}, produktu: {n - len(bg_labels)}")
        if bg_labels:
            bg_mask = np.isin(labeled, list(bg_labels))
            arr[bg_mask, 3] = 0

    arr[transparent, 3] = 0

    fg = Image.fromarray(arr, "RGBA")
    opaque = (arr[:,:,3] > 0).sum()
    print(f"Piksele produktu po usunięciu tła: {opaque}")

    canvas = Image.new("RGB", IMG_SIZE, (0, 0, 0))
    mw, mh = IMG_SIZE[0]-PAD*2, IMG_SIZE[1]-PAD*2
    r = fg.width / fg.height
    nw = mw if r >= 1 else max(1, int(mh*r))
    nh = max(1, int(nw/r)) if r >= 1 else mh
    fgr = fg.resize((nw, nh), Image.LANCZOS)
    canvas.paste(fgr, ((IMG_SIZE[0]-nw)//2, (IMG_SIZE[1]-nh)//2), fgr.split()[3])
    return canvas

# Pobierz pierwszy produkt z repair_list (który miał białe tło)
with open(f"{BASE_DIR}/repair_list.json") as f:
    repair = json.load(f)

# Znajdź pierwszy z białym tłem
tested = None
for p in repair[:20]:
    url = p.get("url","")
    if url:
        r = requests.get(url, headers=WEB_HDR, timeout=15)
        if r.status_code == 200:
            arr_test = np.array(Image.open(io.BytesIO(r.content)).convert("RGBA"))
            h,w = arr_test.shape[:2]
            es = max(4,h//20)
            edge = np.concatenate([arr_test[:es,:,:3].reshape(-1,3),arr_test[-es:,:,:3].reshape(-1,3),
                                   arr_test[:,:es,:3].reshape(-1,3),arr_test[:,-es:,:3].reshape(-1,3)])
            wc = ((edge[:,0]>=218)&(edge[:,1]>=218)&(edge[:,2]>=218)).sum()
            if wc/len(edge) > 0.55:
                tested = p
                raw = r.content
                print(f"Testowy produkt: {p['_id']}, url: {url[:60]}")
                break

if tested:
    result = make_black_bg_fixed(raw)
    result.save(f"{BASE_DIR}/test_result.jpg", "JPEG", quality=92)
    print("Zapisano test_result.jpg")
else:
    print("Nie znaleziono produktu z białym tłem w pierwszych 20")


Testowy produkt: product-p0000176, url: https://cdn.sanity.io/images/nzcwegq7/production/64aeb66968a
Obraz: (1200, 1200), alpha=0 px: 0, jasne px: 969603, komponenty: 780
Komponenty tła: 1, produktu: 779
Piksele produktu po usunięciu tła: 481917
Zapisano test_result.jpg
